In [5]:
import sys
 
import nflreadpy as nfl
import pandas as pd

In [6]:
# Filter by only 2025
SEASON = 2025
 
pbp = nfl.load_pbp(seasons=[SEASON]).to_pandas()

In [ ]:
# Build decision dataset for 4th down plays
fourth = pbp[
    (pbp["down"] == 4)
    & (pbp["play_type"].isin(["run", "pass", "punt", "field_goal"]))
    & (pbp["qb_kneel"].fillna(0) != 1)
    & (pbp["qb_spike"].fillna(0) != 1)
].copy()
 
fourth["decision"] = fourth["play_type"].map(
    {"run": "go", "pass": "go", "punt": "punt", "field_goal": "field_goal"}
)
 

In [ ]:
# Break it down by yards-to-go bucket
# Buckets are right-inclusive: 0-3 = up to 3 yds, 3-7 = 4 to 7, 7-10 = 8 to 10, 10+ = 11 or more
fourth["ytg_bucket"] = pd.cut(
    fourth["ydstogo"],
    bins=[0, 3, 7, 10, float("inf")],
    labels=["0-3", "3-7", "7-10", "10+"],
    include_lowest=True,
)

In [13]:
# Number of plays for each bucket / decision
counts = pd.crosstab(fourth["ytg_bucket"], fourth["decision"], margins=True, margins_name="total")
print(f"\n=== {SEASON}: number of decisions by yards-to-go bucket ===")
print(counts)


=== 2025: number of decisions by yards-to-go bucket ===
decision    field_goal   go  punt  total
ytg_bucket                              
0-3                144  617   340   1101
3-7                408  188   584   1180
7-10               227   69   434    730
10+                245   57   684    986
total             1024  931  2042   3997


In [14]:
# Share of decisions within each bucket (rows sum to 100%)
shares = (
    pd.crosstab(fourth["ytg_bucket"], fourth["decision"], normalize="index") * 100
).round(1)
print(f"\n=== {SEASON}: % of decisions within each bucket ===")
print(shares)


=== 2025: % of decisions within each bucket ===
decision    field_goal    go  punt
ytg_bucket                        
0-3               13.1  56.0  30.9
3-7               34.6  15.9  49.5
7-10              31.1   9.5  59.5
10+               24.8   5.8  69.4


In [15]:
# Average yards to go for each decision inside each bucket
avg_in_bucket = (
    fourth.pivot_table(
        index="ytg_bucket", columns="decision", values="ydstogo",
        aggfunc="mean", observed=False,
    )
    .round(2)
)
print(f"\n=== {SEASON}: average yards to go, decision within bucket ===")
print(avg_in_bucket)


=== 2025: average yards to go, decision within bucket ===
decision    field_goal     go   punt
ytg_bucket                          
0-3               2.35   1.55   2.10
3-7               5.48   4.96   5.55
7-10              8.88   9.07   8.98
10+              14.84  15.09  16.18
